In [ ]:
# Scrape results table from Bart Torvik and save as a csv file
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from io import StringIO
import pandas as pd
import time
import os

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://barttorvik.com/team.php?team=UC+San+Diego&year=2026")

WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
time.sleep(3)

page_source = driver.page_source
driver.quit()

tables = pd.read_html(StringIO(page_source))
print(f"Page scraped — {len(tables)} tables found")

Page scraped — 5 tables found


In [5]:
game_log = tables[3].copy()

# Flatten columns
game_log.columns = ['_'.join(col).strip() for col in game_log.columns]
game_log = game_log.iloc[:, :30]

# Rename
game_log.columns = game_log.columns = [
    'date', 'date2', 'home_away', 'opp_rank', 'opp_rank2',
    'opponent', 'opponent_short', 'result', 'result2',
    'record', 'record2', 'wab', 'opp_adjO', 'opp_adjD',
    'off_eff', 'off_efg', 'off_to', 'off_or', 'off_ftr', 'off_2p', 'off_3p',
    'def_eff', 'def_efg', 'def_to', 'def_or', 'def_ftr', 'def_2p', 'def_3p',
    'gamescore', 'plus_minus'
]

# Drop redundant columns
game_log = game_log.drop(columns=['date2', 'opp_rank2', 'opponent_short', 'result2', 'record2'])

# Keep only real game rows
game_log = game_log[game_log['result'].str.startswith(('W,', 'L,'), na=False)]
game_log = game_log.reset_index(drop=True)

# Engineered columns
game_log['game_num'] = range(1, len(game_log) + 1)
game_log['outcome'] = game_log['result'].str[0]
game_log['location'] = game_log['home_away'].map({'H': 'Home', 'A': 'Away', 'N': 'Neutral'})

# Convert numerics
numeric_cols = [
    'opp_adjO', 'opp_adjD', 'wab',
    'off_eff', 'off_efg', 'off_to', 'off_or', 'off_ftr', 'off_2p', 'off_3p',
    'def_eff', 'def_efg', 'def_to', 'def_or', 'def_ftr', 'def_2p', 'def_3p',
    'gamescore', 'plus_minus'
]
for col in numeric_cols:
    game_log[col] = pd.to_numeric(game_log[col], errors='coerce')

# Derived metrics
game_log['net_eff'] = game_log['off_eff'] - game_log['def_eff']
game_log['opp_adj_margin'] = game_log['opp_adjO'] - game_log['opp_adjD']

print(f"{len(game_log)} games cleaned")
print(game_log[['date', 'opponent', 'result', 'location', 'net_eff']].to_string())

34 games cleaned
                date             opponent     result location  net_eff
0   Mon 11-03  11-03             La Verne  W, 105-73      NaN     42.6
1   Sat 11-08  11-08    Houston Christian   W, 78-60      NaN     26.0
2   Wed 11-12  11-12           Fresno St.   W, 78-73      NaN      7.0
3   Sat 11-15  11-15                Idaho   W, 75-67      NaN     11.4
4   Mon 11-24  11-24               Temple   W, 91-76      NaN     21.9
5   Tue 11-25  11-25              Bradley   W, 87-77      NaN     14.3
6   Wed 11-26  11-26               Towson   W, 87-73      NaN     21.8
7   Tue 12-02  12-02               Nevada   L, 76-70      NaN     -9.3
8   Sat 12-06  12-06       Long Beach St.   W, 80-74      NaN      8.9
9   Sat 12-13  12-13               Tulane   W, 93-67      NaN     33.3
10  Tue 12-16  12-16     Loyola Marymount   W, 67-57      NaN     14.1
11  Fri 12-19  12-19            San Diego   L, 82-80      NaN     -2.9
12  Sun 12-28  12-28              Stanton   W, 85-62      Na

In [10]:
game_log.to_csv("data/raw/torvik_game_log_2026.csv", index=False)
print("Saved to data/raw/torvik_game_log_2026.csv")

Saved to data/raw/torvik_game_log_2026.csv
